# День 4. Оптимизаторы и функции потерь

## 1. Введение: от ручного обновления весов к профессиональным инструментам

### 1.1. Мостик от Дня 3

В Дне 3 ты обучал `SimpleMLP` вот таким циклом:

In [ ]:
loss.backward()
with torch.no_grad():
    for p in model.parameters():
        p -= learning_rate * p.grad
model.zero_grad()

Это **ванильный градиентный спуск (Vanilla SGD)** — тот же алгоритм, что ты реализовывал вручную в Дне 1 и 2, только теперь применённый ко всем параметрам сети через `.parameters()`. Он работает, но у него есть фундаментальные проблемы, которые ты наверняка заметил, если внимательно смотрел на кривые обучения:

- **Один и тот же `learning_rate` для всех параметров.** У одних весов градиент может быть маленьким и стабильным, у других — большим и шумным. Единая скорость обучения — это компромисс, который никогда не оптимален для всех параметров одновременно.
- **Никакой «памяти» о предыдущих шагах.** Если поверхность функции потерь — узкий овраг (типичная ситуация в реальных сетях), ванильный GD будет колебаться от стенки к стенке оврага, а не двигаться вдоль его дна.
- **Чувствительность к выбору `learning_rate`.** Слишком большой — расходимость (`loss = NaN`). Слишком маленький — обучение может занять тысячи эпох.

**`torch.optim`** — это модуль, инкапсулирующий продвинутые алгоритмы обновления весов (SGD с моментумом, Adam и другие), которые решают эти проблемы математически обоснованными способами, а не эвристиками "подбери lr получше".

### 1.2. Цель дня

После этого конспекта ты должен уметь:
- Объяснить физический смысл momentum в SGD и почему он ускоряет сходимость.
- Вывести и объяснить формулы Adam, включая bias correction.
- Осознанно выбирать между SGD и Adam для конкретной задачи.
- Настраивать `scheduler` для уменьшения `learning_rate` по ходу обучения.
- Защищать сеть от взрыва градиентов через `gradient clipping`.
- Понимать, почему `CrossEntropyLoss` и `BCEWithLogitsLoss` численно стабильнее «ручных» реализаций через `softmax`/`sigmoid` + `log`.
- Разделять понятия **loss** (то, что оптимизируется) и **метрика** (то, что интересует бизнес).

## 2. `torch.optim.SGD`: momentum, Nesterov, weight_decay

### 2.1. Базовый SGD — то, что ты уже писал руками

In [ ]:
import torch.optim as optim

optimizer = optim.SGD(model.parameters(), lr=0.01)

# Цикл обучения:
optimizer.zero_grad()   # эквивалент model.zero_grad() из Дня 3
loss.backward()
optimizer.step()         # эквивалент "for p in model.parameters(): p -= lr * p.grad"

`optimizer.step()` без дополнительных параметров реализует ровно то же самое обновление, что ты писал вручную:

In [ ]:
θ_t = θ_{t-1} - lr · ∇L(θ_{t-1})

**Ключевое отличие от твоего ручного кода:** `optimizer` хранит **ссылки** на параметры модели (переданные при создании через `model.parameters()`), а не копии — поэтому `optimizer.step()` обновляет их «на месте» через собственный внутренний `torch.no_grad()`-контекст. Тебе больше не нужно вручную оборачивать обновление в `with torch.no_grad()` и вручную перебирать параметры.

### 2.2. Momentum — физическая аналогия и математика

**Проблема без momentum:** представь функцию потерь как узкий извилистый овраг. Градиент в каждой точке указывает направление наибольшего роста loss — но в узком овраге это направление часто почти перпендикулярно «дну» оврага (туда, куда нужно двигаться), а не вдоль него. Результат — зигзагообразное движение с медленным прогрессом вдоль основного направления.

In [ ]:
Без momentum:              С momentum:

   ↗↙↗↙↗↙  (зигзаги)          ->->->->->  (плавное движение
   ────────-> (медленный          вдоль дна оврага,
              прогресс)           накопленная "скорость")

**Физическая аналогия:** momentum — это как шарик, катящийся по оврагу. У него есть **инерция** (накопленная «скорость» из предыдущих шагов), которая гасит колебания поперёк оврага (они постоянно меняют знак и взаимно уничтожаются при усреднении) и усиливает движение вдоль оврага (оно стабильно в одном направлении на всех шагах).

**Формула (классический momentum, PyTorch):**

In [ ]:
v_t = momentum · v_{t-1} + g_t
θ_t = θ_{t-1} - lr · v_t

где `g_t = ∇L(θ_{t-1})` — градиент на текущем шаге, `v_t` — «скорость» (экспоненциально взвешенное скользящее среднее градиентов), `momentum` — коэффициент затухания (обычно `0.9`).

In [ ]:
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

**Разворачивание рекурсии — почему это скользящее среднее с экспоненциальными весами:**

In [ ]:
v_1 = g_1
v_2 = momentum·g_1 + g_2
v_3 = momentum²·g_1 + momentum·g_2 + g_3
...
v_t = Σ_{k=1}^{t} momentum^{t-k} · g_k

Чем «старше» градиент, тем меньше его вклад (домножается на убывающую степень `momentum`). При `momentum=0.9` вклад градиента 10-шаговой давности — `0.9^10 ≈ 0.35`, а 50-шаговой — `0.9^50 ≈ 0.005` (практически забыт). Это и есть «инерция с забыванием».

### 2.3. Nesterov Accelerated Gradient (NAG)

**Идея:** обычный momentum сначала считает градиент в текущей точке, потом делает шаг с учётом накопленной скорости. Nesterov Momentum сначала «заглядывает вперёд» — считает градиент в точке, куда бы шарик докатился по инерции, а потом корректирует шаг с учётом этого «предвиденного» градиента.

In [ ]:
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, nesterov=True)

**Аналогия:** обычный momentum — это шарик, который катится и «слепо» реагирует на градиент там, где он есть *сейчас*. Nesterov — это более «умный» шарик, который сначала прикидывает, куда его унесёт инерция, и уже там смотрит на градиент — если там подъём, он начинает тормозить чуть раньше, чем обычный momentum. На практике NAG часто чуть быстрее сходится и меньше «проскакивает» минимум.

### 2.4. `weight_decay` — L2-регуляризация как параметр оптимизатора

Ты уже разбирал L2/Ridge-регуляризацию в контексте логистической регрессии (Неделя 4 твоего roadmap): штраф `λ·Σw²` добавляется к функции потерь, чтобы веса не росли бесконтрольно, снижая дисперсию модели ценой небольшого роста смещения (Bias-Variance Tradeoff).

В нейросетях это делается **не** явным изменением `loss`, а параметром `weight_decay` внутри оптимизатора:

In [ ]:
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)

**Математически это эквивалентно** добавлению члена `λ·θ` прямо к градиенту перед шагом обновления:

In [ ]:
g_t = ∇L(θ_{t-1}) + weight_decay · θ_{t-1}
θ_t = θ_{t-1} - lr · g_t

Это ровно то же самое, что добавить в loss слагаемое `(weight_decay/2)·||θ||²` и взять от него градиент (`d/dθ [(λ/2)θ²] = λθ`) — то есть классический L2-penalty из Ridge-регрессии, только применённый к весам сети, а не к коэффициентам линейной модели.

**Важный нюанс (важно для Дня 6):** для чистого SGD `weight_decay`, реализованный как «добавка к градиенту», математически эквивалентен L2-регуляризации. Но для Adam эта эквивалентность **нарушается** из-за адаптивного масштабирования шага (раздел 3) — отсюда разница между `Adam` (weight_decay через градиент) и `AdamW` (weight_decay применяется отдельно, «правильно» — это будет важно в Дне 6 при обучении `TabularMLP`).

### 2.5. Полная сигнатура `torch.optim.SGD`

In [ ]:
optimizer = optim.SGD(
    model.parameters(),
    lr=0.01,          # learning rate — обязательный параметр
    momentum=0.9,      # коэффициент инерции (0 = ванильный SGD)
    nesterov=True,     # Nesterov-версия momentum (требует momentum > 0)
    weight_decay=1e-4  # L2-регуляризация
)

## 3. `torch.optim.Adam`: адаптивный learning rate

### 3.1. Мотивация — почему единый `learning_rate` неоптимален

Представь сеть с двумя параметрами: `w1` получает частые большие градиенты, `w2` — редкие маленькие (типичная ситуация для эмбеддингов или разреженных признаков — вспомни разреженные one-hot фичи из твоего FraudGuard). Единый `lr` для обоих:
- либо слишком большой для `w1` -> колебания/расходимость,
- либо слишком маленький для `w2` -> крайне медленное обучение.

**Идея Adam (Adaptive Moment Estimation):** поддерживать для **каждого параметра отдельно** два скользящих статистических момента градиента и использовать их для адаптации эффективного шага индивидуально.

### 3.2. Первый момент (m) — направление, как momentum

In [ ]:
m_t = β₁ · m_{t-1} + (1-β₁) · g_t

Это экспоненциальное скользящее среднее градиента — концептуально то же самое, что `v_t` в SGD с momentum (раздел 2.2), только с другой нормировкой (`(1-β₁)` вместо `1`). По умолчанию `β₁ = 0.9`.

### 3.3. Второй момент (v) — «энергия» градиента, для адаптивного масштаба

In [ ]:
v_t = β₂ · v_{t-1} + (1-β₂) · g_t²

Это экспоненциальное скользящее среднее **квадрата** градиента (не путать с дисперсией — среднее не вычитается, это «второй момент относительно нуля», иногда называется uncentered variance). По умолчанию `β₂ = 0.999` (растягивается на более долгую историю, чем `β₁`).

**Зачем нужен квадрат градиента:** он даёт оценку **масштаба** (величины, «энергии») колебаний градиента для данного параметра, независимо от знака. Если параметр получает стабильно большие по модулю градиенты (не важно, положительные или отрицательные), `v_t` будет большим.

### 3.4. Bias correction — почему нужна коррекция смещения

При инициализации `m_0 = 0`, `v_0 = 0`. На первых шагах (маленькие `t`) эти скользящие средние сильно смещены к нулю — «холодный старт».

**Пример при `t=1`:**

In [ ]:
m_1 = β₁·0 + (1-β₁)·g_1 = (1-0.9)·g_1 = 0.1·g_1

Реальный градиент `g_1`, а оценка `m_1` — всего `0.1·g_1`, то есть **в 10 раз меньше** истинного значения. Без коррекции первые шаги обучения были бы искусственно замедлены.

**Решение — bias-corrected оценки:**

In [ ]:
m̂_t = m_t / (1 - β₁^t)
v̂_t = v_t / (1 - β₂^t)

При `t=1`: `1 - β₁^1 = 1 - 0.9 = 0.1`, значит `m̂_1 = 0.1·g_1 / 0.1 = g_1` — коррекция полностью восстанавливает истинный масштаб градиента на первом шаге. По мере роста `t`, `β₁^t -> 0`, значит `(1-β₁^t) -> 1`, и коррекция становится всё менее значимой (на поздних шагах `m̂_t ≈ m_t`).

### 3.5. Итоговое правило обновления Adam

In [ ]:
θ_t = θ_{t-1} - lr · m̂_t / (√v̂_t + ε)

`ε` (обычно `1e-8`) — защита от деления на ноль, если `v̂_t` близко к нулю (аналогичный паттерн клиппинга ты уже применял для `log(0)` в ручной реализации `compute_log_loss` в Неделе 4).

**Почему это «адаптивный learning rate» для каждого параметра:**
- Если у параметра исторически **большие** градиенты -> `v̂_t` большое -> `√v̂_t` большое -> эффективный шаг `lr / √v̂_t` **уменьшается** (тормозим параметр, который и так быстро меняется).
- Если у параметра исторически **маленькие/редкие** градиенты -> `v̂_t` маленькое -> эффективный шаг **увеличивается** (даём параметру «догнать» остальные).

**Практическое следствие:** Adam особенно хорош там, где разные параметры получают сигнал очень разной частоты и амплитуды — например, эмбеддинги (некоторые категории редкие, другие частые) или разреженные табличные признаки. Это одна из причин, почему Adam — стандартный выбор по умолчанию для табличных нейросетей вроде твоего будущего `TabularMLP`.

### 3.6. Использование в PyTorch

In [ ]:
optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3,               # типичный дефолт для Adam (для SGD обычно нужен больше — 0.01-0.1)
    betas=(0.9, 0.999),    # (β₁, β₂)
    eps=1e-8,
    weight_decay=0         # см. предупреждение раздела 2.4 про AdamW
)

**Практическое правило:** `lr=1e-3` — хороший старт для Adam почти в любой задаче. Для SGD этот же `lr` часто *слишком мал* — задача 12.5 практики этого дня продемонстрирует это явно.

### 3.7. Когда SGD, когда Adam — сравнительная таблица

| Критерий | SGD (+ momentum) | Adam |
|:---|:---|:---|
| Скорость сходимости на старте | Медленнее, чувствителен к `lr` | Обычно быстрее «из коробки» |
| Устойчивость к выбору гиперпараметров | Требует тщательного тюнинга `lr`, `momentum` | Устойчив к дефолтным значениям — хороший выбор для быстрого прототипирования |
| Память на параметр | 1 доп. буфер (`v` для momentum) | 2 доп. буфера (`m` и `v`) — в 2 раза больше памяти на состояние оптимизатора |
| Обобщающая способность (generalization) | Часто лучше на классических CV-задачах (ResNet и др. классически обучаются SGD+momentum) | Иногда сходится к «острым» минимумам с чуть худшим обобщением на очень больших моделях (не критично для табличных MLP) |
| Табличные данные / разреженные фичи | Хуже справляется с редкими сигналами | Лучше — адаптивный масштаб на параметр |
| **Практический дефолт для этого курса** | — | **Adam** (или `AdamW` в Дне 6) |

## 4. `Scheduler`: управление `learning_rate` по ходу обучения

### 4.1. Зачем вообще менять `lr` во время обучения

**Интуиция:** в начале обучения веса далеки от оптимума — нужны большие шаги, чтобы быстро приблизиться к «хорошей» области. Ближе к концу обучения веса уже около минимума — большие шаги начинают «перескакивать» через него, вызывая колебания loss около хорошего, но не оптимального значения. Уменьшение `lr` со временем позволяет сделать финальную «тонкую доводку».

In [ ]:
Большой lr:  loss колеблется вокруг минимума, не может "успокоиться"
Малый lr:    loss плавно оседает в минимум, но медленно
Оптимально:  большой lr вначале -> уменьшение по ходу обучения

### 4.2. `StepLR` — ступенчатое уменьшение

In [ ]:
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

**Формула:**

In [ ]:
lr_epoch = lr_0 · gamma^⌊epoch / step_size⌋

При `step_size=5`, `gamma=0.5`, `lr_0=1e-3`:

In [ ]:
Эпохи 0-4:   lr = 1e-3
Эпохи 5-9:   lr = 5e-4
Эпохи 10-14: lr = 2.5e-4

**Порядок вызова — критично!** `scheduler.step()` вызывается **после** `optimizer.step()`, **один раз за эпоху** (не за батч):

In [ ]:
for epoch in range(n_epochs):
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()       # обновление весов — за КАЖДЫЙ батч

    scheduler.step()            # обновление lr — за КАЖДУЮ эпоху (после всех батчей)

### 4.3. `ReduceLROnPlateau` — адаптивное уменьшение по метрике

В отличие от `StepLR` (расписание фиксировано заранее), `ReduceLROnPlateau` следит за метрикой (обычно `val_loss`) и уменьшает `lr`, если метрика **перестала улучшаться** в течение `patience` эпох подряд.

In [ ]:
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

for epoch in range(n_epochs):
    train_one_epoch(...)
    val_loss = evaluate(...)

    scheduler.step(val_loss)   # ВАЖНО: сюда передаётся метрика, в отличие от StepLR!

**Параметры:**
- `mode='min'` — метрика должна уменьшаться (для loss). `mode='max'` — для метрик вроде accuracy.
- `factor=0.5` — во сколько раз уменьшить `lr` при срабатывании (`lr_new = lr_old * factor`).
- `patience=3` — сколько эпох ждать без улучшения, прежде чем сработать.

**Частая ошибка:** `scheduler.step()` без аргумента для `ReduceLROnPlateau` — упадёт с `TypeError`, потому что этому шедулеру **обязательно** нужна метрика для принятия решения (в отличие от `StepLR`, у которого `.step()` работает по расписанию и метрика не нужна).

### 4.4. Сравнительная таблица шедулеров

| Scheduler | Логика | Когда использовать |
|:---|:---|:---|
| `StepLR` | Фиксированное расписание (каждые N эпох ×gamma) | Когда заранее знаешь примерную динамику обучения (из прошлых экспериментов) |
| `ExponentialLR` | `lr *= gamma` **каждую** эпоху (непрерывное затухание) | Плавное затухание без «ступенек» |
| `ReduceLROnPlateau` | Реагирует на застой метрики | Когда не знаешь заранее динамику — самый «безопасный» дефолт |
| `CosineAnnealingLR` | Косинусное затухание от `lr_max` до `lr_min` | Часто используется в современных SOTA-рецептах (за пределами этого курса, но полезно знать название) |

## 5. Gradient Clipping — защита от взрыва градиентов

### 5.1. Проблема exploding gradients

В глубоких сетях (особенно рекуррентных, но встречается и в MLP при неудачной инициализации или большом `lr`) градиенты, проходя через chain rule (День 2) по многим слоям, могут **экспоненциально расти**. Результат — веса «взрываются» до огромных значений или `NaN`.

**Диагностика:** посмотри на норму градиента:

In [ ]:
total_norm = torch.norm(torch.stack([p.grad.norm() for p in model.parameters() if p.grad is not None]))
print(total_norm.item())

Если это число растёт от эпохи к эпохе без остановки (или внезапно скачет на порядки) — это exploding gradients.

### 5.2. `clip_grad_norm_` — обрезка по общей норме

In [ ]:
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()

**Что происходит:** считается **общая норма** (обычно L2-норма) вектора, составленного из всех градиентов всех параметров вместе. Если эта норма превышает `max_norm`, **все** градиенты домножаются на один и тот же масштабирующий коэффициент, чтобы итоговая общая норма стала равна `max_norm`:

In [ ]:
total_norm = ||[g_1, g_2, ..., g_n]||₂    (конкатенация ВСЕХ градиентов в один вектор)

если total_norm > max_norm:
    scale = max_norm / total_norm
    для каждого g_i: g_i = g_i · scale

**Ключевое свойство:** масштабирование единое для всех параметров — сохраняется **направление** суммарного градиента, меняется только его длина. Это отличает `clip_grad_norm_` от `clip_grad_value_`.

### 5.3. `clip_grad_value_` — обрезка поэлементно

In [ ]:
torch.nn.utils.clip_grad_value_(model.parameters(), clip_value=0.5)

Каждый элемент градиента независимо обрезается в диапазон `[-clip_value, clip_value]`:

In [ ]:
g_i = max(-clip_value, min(clip_value, g_i))

**Отличие от `clip_grad_norm_`:** здесь **направление** суммарного градиента может измениться (потому что каждая компонента обрезается независимо), в отличие от нормы, где обрезка — это равномерное масштабирование. На практике `clip_grad_norm_` используется чаще, потому что сохраняет геометрию градиента.

### 5.4. Порядок вызова — критично

In [ ]:
optimizer.zero_grad()
loss.backward()                                              # 1. посчитать градиенты
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)      # 2. обрезать градиенты
optimizer.step()                                               # 3. обновить веса обрезанными градиентами

Если вызвать clipping **до** `.backward()` или **после** `optimizer.step()` — эффекта не будет вообще (клиппинг работает с уже посчитанными `.grad`, которые до `backward()` пусты/устарели, а после `step()` веса уже обновлены).

## 6. Функции потерь: математика и численная стабильность

### 6.1. `nn.MSELoss` — для регрессии

In [ ]:
MSE = (1/N) · Σ (y_pred_i - y_true_i)²

In [ ]:
criterion = nn.MSELoss()
loss = criterion(y_pred, y_true)   # оба тензора одинаковой формы

Это та же формула, что ты писал вручную в Дне 1 (`(diff ** 2).mean()`). Ничего нового математически — просто готовый, оптимизированный модуль.

### 6.2. `nn.CrossEntropyLoss` — многоклассовая классификация

#### 6.2.1. Математика: Softmax + Negative Log-Likelihood

Для задачи с `C` классами модель выдаёт вектор логитов `z = (z_1, ..., z_C)`. Softmax превращает его в распределение вероятностей:

In [ ]:
Softmax(z)_c = exp(z_c) / Σ_{j=1}^{C} exp(z_j)

Если истинный класс — `c*`, кросс-энтропия для одного примера:

In [ ]:
L = -log( Softmax(z)_{c*} )
  = -z_{c*} + log( Σ_j exp(z_j) )

Второе слагаемое — `log-sum-exp` (LSE) от логитов.

#### 6.2.2. Worked example — считаем вручную

3 класса, логиты `z = [2.0, 1.0, 0.1]`, истинный класс `c* = 0`.

In [ ]:
exp(2.0) = 7.389
exp(1.0) = 2.718
exp(0.1) = 1.105
Σ = 11.212

Softmax(z)_0 = 7.389 / 11.212 = 0.6590
Softmax(z)_1 = 2.718 / 11.212 = 0.2424
Softmax(z)_2 = 1.105 / 11.212 = 0.0986
                                --------
                                1.0000  ✓ (сумма = 1)

L = -log(0.6590) = 0.4170

Проверим в PyTorch:

In [ ]:
import torch
import torch.nn as nn

logits = torch.tensor([[2.0, 1.0, 0.1]])
target = torch.tensor([0])              # индекс класса, НЕ one-hot!

criterion = nn.CrossEntropyLoss()
loss = criterion(logits, target)
print(loss.item())   # 0.4170...  — совпадает с ручным расчётом

#### 6.2.3. Почему `CrossEntropyLoss` = LogSoftmax + NLLLoss «в одном флаконе»

In [ ]:
# То, что делает CrossEntropyLoss внутри (концептуально):
log_probs = torch.log_softmax(logits, dim=1)   # LogSoftmax
loss = torch.nn.functional.nll_loss(log_probs, target)   # Negative Log-Likelihood

# Эквивалентно:
loss = nn.CrossEntropyLoss()(logits, target)

**Почему нельзя (точнее — не нужно и опасно) делать это раздельно вручную через `Softmax` + `log`:**

In [ ]:
# ОПАСНО — не делай так:
probs = torch.softmax(logits, dim=1)
loss = -torch.log(probs[range(len(target)), target]).mean()

Если логиты большие (например, `z = [1000.0, 1.0, 0.1]`), `exp(1000.0)` переполняет `float32` (`inf`), и деление `inf/inf` даёт `NaN`. `CrossEntropyLoss` **не вычисляет** `softmax` явно — она использует **LogSumExp-трюк**, вычисляя `log(Σ exp(z_j))` через вычитание максимума:

In [ ]:
log(Σ exp(z_j)) = max(z) + log(Σ exp(z_j - max(z)))

После вычитания `max(z)` максимальный аргумент экспоненты становится `0` (`exp(0)=1`), а все остальные — отрицательными (`exp(отрицательное) ∈ (0,1)`) — переполнение становится математически невозможным. Это ровно тот же принцип «клиппинга для численной стабильности», который ты применял вручную в `compute_log_loss` для логистической регрессии (Неделя 4 твоего roadmap) — только реализованный на уровне более общей формулы.

#### 6.2.4. Требования к входам — частый источник ошибок

In [ ]:
criterion = nn.CrossEntropyLoss()

logits = torch.randn(32, 10)                      # (batch, num_classes) — СЫРЫЕ логиты, БЕЗ softmax!
target = torch.randint(0, 10, (32,))              # (batch,) — индексы классов, dtype=torch.long

loss = criterion(logits, target)   # OK

| Требование | Правильно | Неправильно |
|:---|:---|:---|
| Форма логитов | `(N, C)` | Применять `softmax` заранее — двойное применение искажает loss |
| Тип `target` | `torch.long` (int64) | `torch.float32` -> `RuntimeError: expected scalar type Long but found Float` |
| Формат `target` | Индексы классов `(N,)`: `[0, 3, 1, ...]` | One-hot `(N, C)`: `[[1,0,0],[0,0,1],...]` — не то, что ждёт этот loss (для one-hot целей нужна другая функция) |

### 6.3. `nn.BCELoss` vs `nn.BCEWithLogitsLoss` — бинарная классификация

#### 6.3.1. Формула Binary Cross-Entropy

Ты уже реализовывал эту формулу вручную (`compute_log_loss`) в контексте логистической регрессии:

In [ ]:
BCE = -(1/N) Σ [ y_i · log(p_i) + (1-y_i) · log(1-p_i) ]

где `p_i ∈ (0, 1)` — предсказанная вероятность (после sigmoid), `y_i ∈ {0, 1}` — истинная метка.

#### 6.3.2. `nn.BCELoss` — ожидает уже готовые вероятности

In [ ]:
sigmoid = nn.Sigmoid()
criterion = nn.BCELoss()

logits = torch.tensor([2.0, -1.0, 0.5])
probs = sigmoid(logits)              # ОБЯЗАТЕЛЬНО применить sigmoid ДО передачи в BCELoss
target = torch.tensor([1.0, 0.0, 1.0])

loss = criterion(probs, target)

**Проблема:** если `probs` в результате округления с плавающей точкой оказывается точно `0.0` или `1.0` (может случиться при экстремальных логитах, например `logits=50.0` -> `sigmoid(50.0) ≈ 1.0` из-за ограничений `float32`), то `log(0)` даёт `-inf`, а loss становится `NaN`. Именно эту проблему ты уже решал вручную клиппингом вероятностей в `compute_log_loss`.

#### 6.3.3. `nn.BCEWithLogitsLoss` — численно стабильная версия

In [ ]:
criterion = nn.BCEWithLogitsLoss()

logits = torch.tensor([2.0, -1.0, 0.5])   # СЫРЫЕ логиты — sigmoid ВНУТРИ loss
target = torch.tensor([1.0, 0.0, 1.0])

loss = criterion(logits, target)   # sigmoid применяется автоматически, стабильным способом

**Формула, которую используют внутри (log-sum-exp вариант для BCE):**

In [ ]:
loss_i = max(x_i, 0) - x_i · y_i + log(1 + exp(-|x_i|))

где `x_i` — сырой логит. Это математически эквивалентно `-[y·log(σ(x)) + (1-y)·log(1-σ(x))]`, но **без явного вычисления `σ(x)`** — а значит, без риска, что `σ(x)` округлится до `0.0` или `1.0` и `log` взорвётся. Слагаемое `exp(-|x_i|)` всегда ограничено сверху единицей (потому что `|x_i| ≥ 0` -> `-|x_i| ≤ 0` -> `exp(-|x_i|) ≤ 1`), что гарантирует отсутствие переполнения при любых значениях логитов.

**Именно поэтому в Дне 3 в `SimpleMLP` выходной слой возвращал сырые логиты**, а не вероятности после sigmoid — это было прямой подготовкой к использованию `BCEWithLogitsLoss` здесь.

#### 6.3.4. `pos_weight` — балансировка классов внутри loss (предвосхищение Дня 6)

In [ ]:
# Если положительный класс в 10 раз реже отрицательного:
pos_weight = torch.tensor([10.0])
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

Это прямой аналог `class_weight='balanced'` из sklearn (который ты уже использовал для LightGBM и логистической регрессии) — штраф за ошибку на положительном (редком) классе домножается на `pos_weight`, заставляя модель уделять ему больше внимания. Это то, что ты применишь в Дне 6 для `Fraud Detection Lite` из-за дисбаланса классов в IEEE-CIS.

### 6.4. Сводная таблица: какой loss для какой задачи

| Задача | Loss | Что подаётся на вход `model` (выходной слой) | Активация внутри loss |
|:---|:---|:---|:---|
| Регрессия | `nn.MSELoss` | Сырое число (без активации) | Нет |
| Бинарная классификация | `nn.BCEWithLogitsLoss` | Сырой логит (1 выход) | Sigmoid (внутри, стабильно) |
| Многоклассовая классификация | `nn.CrossEntropyLoss` | Сырые логиты (`C` выходов) | LogSoftmax (внутри, стабильно) |
| Multi-label классификация (несколько независимых меток) | `nn.BCEWithLogitsLoss` (поэлементно) | Сырые логиты (`C` выходов, независимых) | Sigmoid поэлементно |

**Золотое правило:** модель почти всегда должна возвращать **сырые логиты**. Активация (`sigmoid`/`softmax`) либо встроена в loss (для численной стабильности), либо применяется отдельно только для **интерпретации** результата (получить вероятности для вывода пользователю), но никогда — перед `CrossEntropyLoss`/`BCEWithLogitsLoss`.

## 7. Loss ≠ Метрика: почему это разные вещи

### 7.1. Разграничение понятий

- **Loss (функция потерь)** — то, что **дифференцируемо** и по чему считается градиент. Это математический суррогат того, что нас на самом деле интересует, подобранный так, чтобы его можно было оптимизировать через `backward()`.
- **Метрика** — то, что реально волнует бизнес/задачу (Accuracy, Precision, Recall, F1, ROC-AUC, PR-AUC — ты уже прорабатывал их все в своём ML-курсе метрик). Метрика часто **не дифференцируема** или недифференцируема в «полезных» точках.

**Пример несовместимости:** Accuracy — это доля правильных предсказаний. Её график по параметрам модели — почти всюду **плоский**, со скачками только в точках, где предсказание переходит через порог `0.5`. Производная почти всюду равна `0` — градиентный спуск **не может** напрямую оптимизировать Accuracy (нет сигнала, куда двигаться). Именно поэтому мы оптимизируем `CrossEntropyLoss`/`BCEWithLogitsLoss` (гладкие, дифференцируемые суррогаты), а Accuracy **вычисляем отдельно**, только для мониторинга.

### 7.2. Практическое следствие: loss может падать, а метрика — не расти линейно с ним

Loss и метрика коррелируют, но не обязаны двигаться строго синхронно на каждом шаге. Небольшое падение `CrossEntropyLoss` может не изменить Accuracy вообще (если ни одно предсказание не «перепрыгнуло» через порог принятия решения), а иногда Accuracy может даже кратковременно чуть просесть, пока loss продолжает падать (модель может стать более «уверенной» в неправильном предсказании, увеличивая абсолютную величину его вклада в loss, при этом граница решения ещё не поменялась).

### 7.3. Вычисление метрик вручную во время обучения (без вызова sklearn на каждый батч)

Вызывать `sklearn.metrics.f1_score(...)` на каждом батче внутри цикла обучения — расточительно (перенос тензоров на CPU, конвертация в NumPy на каждой итерации создаёт заметный оверхед при частом вызове). Для мониторинга **во время обучения** метрики считают прямо на тензорах:

In [ ]:
def compute_accuracy(logits, y_true):
    """Многоклассовая accuracy: argmax по логитам."""
    preds = logits.argmax(dim=1)
    return (preds == y_true).float().mean().item()

def compute_binary_f1(logits, y_true, threshold=0.5):
    """Бинарный F1 вручную, без sklearn — для быстрого мониторинга внутри батча."""
    with torch.no_grad():
        probs = torch.sigmoid(logits)
        preds = (probs >= threshold).float()

        tp = ((preds == 1) & (y_true == 1)).sum().float()
        fp = ((preds == 1) & (y_true == 0)).sum().float()
        fn = ((preds == 0) & (y_true == 1)).sum().float()

        precision = tp / (tp + fp + 1e-8)     # +eps — тот же паттерн защиты от деления на 0
        recall = tp / (tp + fn + 1e-8)
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        return f1.item()

**Практика в этом курсе:** «быстрые» метрики (Accuracy) считаем вручную на тензорах на **каждой** эпохе для логов прогресса. Более тяжёлые/детальные метрики (полная Confusion Matrix, PR-AUC, ROC-AUC — уже знакомые тебе из sklearn) считаем через `sklearn.metrics` **в конце обучения** или **раз в несколько эпох** на полном val-сете, конвертировав тензоры в NumPy один раз (`.detach().cpu().numpy()`) — это ты уже будешь делать в Дне 6 для `Fraud Detection Lite`.

## 8. Практика: полный цикл обучения на Fashion-MNIST

### 8.1. Что нового в практике (мостик к Дню 5)

Этот раздел впервые использует `torchvision.datasets` и `DataLoader` — полноценный разбор `Dataset`/`DataLoader` (кастомные датасеты, `collate_fn`, `num_workers`) будет в Дне 5. Сегодня используем их **в готовом виде** — как «чёрный ящик», который отдаёт батчи `(X_batch, y_batch)` — чтобы сфокусироваться именно на оптимизаторах и loss.

### 8.2. Полный код

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используем device: {device}")

# ============================================
# 1. Данные
# ============================================

transform = transforms.Compose([
    transforms.ToTensor(),   # PIL Image (0-255) -> Tensor (0.0-1.0), форма (1, 28, 28)
])

train_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

classes = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
           'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f"Train: {len(train_dataset)} примеров, Test: {len(test_dataset)} примеров")

# Посмотрим на форму одного батча
X_sample, y_sample = next(iter(train_loader))
print(f"Форма батча X: {X_sample.shape}")   # torch.Size([64, 1, 28, 28])
print(f"Форма батча y: {y_sample.shape}")   # torch.Size([64])

# ============================================
# 2. Модель
# ============================================

class FashionMLP(nn.Module):
    """MLP: Flatten -> Linear(784,256) -> ReLU -> Linear(256,128) -> ReLU -> Linear(128,10)"""

    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()   # (N, 1, 28, 28) -> (N, 784)
        self.net = nn.Sequential(
            nn.Linear(28 * 28, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)       # 10 логитов, БЕЗ softmax — CrossEntropyLoss применит его сам
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.net(x)

def make_model():
    torch.manual_seed(0)   # одинаковая инициализация для честного сравнения оптимизаторов (раздел 8.4)
    return FashionMLP().to(device)

model = make_model()
n_params = sum(p.numel() for p in model.parameters())
print(f"Параметров в модели: {n_params:,}")
# Linear(784,256): 784*256+256 = 200_960
# Linear(256,128): 256*128+128 = 32_896
# Linear(128,10):  128*10+10   = 1_290
# Итого: 235_146

# ============================================
# 3. Loss, Optimizer, Scheduler
# ============================================

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

# ============================================
# 4. Вспомогательные функции
# ============================================

def compute_accuracy(logits, labels):
    preds = logits.argmax(dim=1)
    return (preds == labels).float().mean().item()

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, total_acc, n_batches = 0.0, 0.0, 0

    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_acc += compute_accuracy(logits, y_batch)
        n_batches += 1

    return total_loss / n_batches, total_acc / n_batches

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_acc, n_batches = 0.0, 0.0, 0

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            total_loss += loss.item()
            total_acc += compute_accuracy(logits, y_batch)
            n_batches += 1

    return total_loss / n_batches, total_acc / n_batches

# ============================================
# 5. Основной цикл обучения — 10 эпох
# ============================================

n_epochs = 10
history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': [], 'lr': []}

for epoch in range(n_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    current_lr = optimizer.param_groups[0]['lr']   # текущий lr — читаем ДО scheduler.step()
    scheduler.step()                                  # StepLR: без аргумента, раз в эпоху

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)
    history['lr'].append(current_lr)

    print(f"Эпоха {epoch+1:2d}/{n_epochs} | lr={current_lr:.5f} | "
          f"train_loss={train_loss:.4f} acc={train_acc:.4f} | "
          f"test_loss={test_loss:.4f} acc={test_acc:.4f}")

# ============================================
# 6. Визуализация
# ============================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['train_loss'], label='train')
axes[0].plot(history['test_loss'], label='test')
axes[0].set_title('Loss (CrossEntropy)')
axes[0].set_xlabel('Эпоха')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history['train_acc'], label='train')
axes[1].plot(history['test_acc'], label='test')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Эпоха')
axes[1].legend()
axes[1].grid(True)

axes[2].plot(history['lr'], marker='o')
axes[2].set_title('Learning Rate (StepLR)')
axes[2].set_xlabel('Эпоха')
axes[2].set_yscale('log')
axes[2].grid(True)

plt.tight_layout()
plt.savefig('day4_fashion_mnist_training.png', dpi=150)
plt.show()

### 8.3. Ожидаемое поведение

- `train_loss`/`test_loss` должны монотонно (в среднем) падать, `train_acc`/`test_acc` — расти до `~87-89%` к 10-й эпохе (это типичный результат для простого MLP на Fashion-MNIST — сверточные сети дают выше, но это тема за пределами этого курса).
- На графике `learning_rate` должна быть видна характерная «лестница» — ступенчатое падение `lr` на 5-й эпохе (в 2 раза, согласно `gamma=0.5`).
- Небольшой разрыв между `train_acc` и `test_acc` — нормален; если разрыв большой и растёт — сигнал переобучения (вспомни Bias-Variance Tradeoff и Dropout из Дня 3 как инструмент борьбы с этим).

### 8.4. Сравнение Adam vs SGD — скорость сходимости за первые 3 эпохи

In [ ]:
def train_few_epochs(optimizer_name, optimizer_fn, n_epochs=3):
    """optimizer_fn: функция, которая принимает model.parameters() и возвращает optimizer"""
    model = make_model()   # ОДИНАКОВАЯ инициализация (torch.manual_seed(0) внутри make_model)
    optimizer = optimizer_fn(model.parameters())
    criterion = nn.CrossEntropyLoss()

    losses = []
    for epoch in range(n_epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        losses.append(train_loss)
        print(f"[{optimizer_name}] Эпоха {epoch+1}: loss={train_loss:.4f} acc={train_acc:.4f}")
    return losses

print("=== Adam (lr=1e-3) ===")
losses_adam = train_few_epochs(
    'Adam', lambda params: optim.Adam(params, lr=1e-3)
)

print("\n=== SGD без momentum (lr=1e-3) ===")
losses_sgd_plain = train_few_epochs(
    'SGD (без momentum)', lambda params: optim.SGD(params, lr=1e-3)
)

print("\n=== SGD с momentum (lr=1e-3, momentum=0.9) ===")
losses_sgd_momentum = train_few_epochs(
    'SGD+momentum', lambda params: optim.SGD(params, lr=1e-3, momentum=0.9)
)

# Визуализация сравнения
plt.figure(figsize=(8, 5))
plt.plot(losses_adam, marker='o', label='Adam (lr=1e-3)')
plt.plot(losses_sgd_plain, marker='s', label='SGD без momentum (lr=1e-3)')
plt.plot(losses_sgd_momentum, marker='^', label='SGD+momentum (lr=1e-3)')
plt.xlabel('Эпоха')
plt.ylabel('Train Loss')
plt.title('Сравнение оптимизаторов: скорость сходимости за 3 эпохи')
plt.legend()
plt.grid(True)
plt.savefig('day4_optimizer_comparison.png', dpi=150)
plt.show()

**Ожидаемый результат и его интерпретация:**

- **Adam** обычно уже за 1-2 эпохи резко снижает loss (адаптивный масштаб шага «сам» находит подходящую эффективную скорость для каждого параметра).
- **SGD без momentum** с тем же `lr=1e-3` будет заметно медленнее — это `lr`, подобранный «под Adam», а не под vanilla SGD. Это специально показывает: **гиперпараметры не переносятся между оптимизаторами напрямую** — то, что хорошо для Adam, может быть слишком мало (или много) для SGD.
- **SGD+momentum** с тем же `lr` обычно немного быстрее чистого SGD, но всё ещё уступает Adam на этом горизонте в 3 эпохи — momentum помогает, но не решает проблему единого `lr` для всех параметров (раздел 3.1).

**Вывод для практики:** если нужно быстро сделать baseline без долгого тюнинга — начинай с Adam и `lr=1e-3`. Если переходишь на SGD (например, из соображений лучшего обобщения на больших моделях за пределами табличных данных) — закладывай на подбор `lr` отдельное время, типичные значения на порядок-два больше, чем для Adam (`0.01`-`0.1`).

## 9. Типичные ошибки Дня 4

| Ошибка | Причина | Решение |
|:---|:---|:---|
| Loss не падает или скачет случайным образом | Забыл `optimizer.zero_grad()` — градиенты накапливаются между батчами | Всегда `optimizer.zero_grad()` перед `loss.backward()` на каждой итерации |
| `RuntimeError: expected scalar type Long but found Float` | `target` для `CrossEntropyLoss` передан как `float`, а не `long` | `y_batch.long()` или изначально создавай метки через `torch.tensor(..., dtype=torch.long)` |
| Loss необычно высокий / плохая сходимость с `CrossEntropyLoss` | Применил `softmax`/`log_softmax` к логитам ДО передачи в `CrossEntropyLoss` — двойное применение | Передавай **сырые** логиты — активация уже встроена в loss |
| Loss = `NaN` при использовании `BCELoss` | Вероятности округлились до точного `0.0`/`1.0` из-за экстремальных логитов | Используй `BCEWithLogitsLoss` вместо `Sigmoid()` + `BCELoss()` |
| `TypeError: step() missing 1 required positional argument: 'metrics'` | `ReduceLROnPlateau.step()` вызван без аргумента метрики | `scheduler.step(val_loss)`, а не `scheduler.step()` — в отличие от `StepLR` |
| `lr` не меняется вообще, хотя `scheduler.step()` вызывается | `scheduler.step()` вызывается **внутри** цикла по батчам вместо цикла по эпохам (для `StepLR`) | Проверь, что `step_size` считается в **эпохах**, а вызов — раз за эпоху, а не за батч |
| Gradient Clipping «не работает» | Вызван до `loss.backward()` или после `optimizer.step()` | Порядок строго: `backward()` -> `clip_grad_norm_()` -> `optimizer.step()` |
| SGD учится в разы медленнее Adam при одинаковом `lr` | Гиперпараметры Adam и SGD не взаимозаменяемы — у SGD обычно нужен `lr` на порядок больше | Подбирай `lr` отдельно под каждый оптимизатор, не копируй значение бездумно |
| Метрика (Accuracy) не растёт, хотя loss падает | Нормальная ситуация вблизи порога решения — loss может «уточнять уверенность», не меняя итоговый класс | Не паникуй сразу; смотри на loss как на основной сигнал сходимости, метрику — как на бизнес-индикатор |

## 10. Чек-лист навыков Дня 4

| Навык | Проверь себя |
|:---|:---|
| Объяснить, зачем нужен `torch.optim`, если можно обновлять веса вручную |  |
| Вывести формулу SGD с momentum и объяснить физическую аналогию (шарик в овраге) |  |
| Объяснить, что делает `weight_decay` математически (связь с L2/Ridge) |  |
| Вывести формулы первого и второго момента Adam (`m_t`, `v_t`) |  |
| Объяснить, зачем нужна bias correction в Adam и что происходит без неё |  |
| Сформулировать, когда выбрать SGD, а когда Adam |  |
| Настроить `StepLR` и объяснить разницу с `ReduceLROnPlateau` |  |
| Объяснить, зачем вызывается `scheduler.step()`, и в каком порядке относительно `optimizer.step()` |  |
| Применить `clip_grad_norm_` в правильном месте цикла обучения |  |
| Вывести формулу CrossEntropyLoss и посчитать её вручную на маленьком примере |  |
| Объяснить LogSumExp-трюк и почему он предотвращает переполнение |  |
| Объяснить разницу `BCELoss` vs `BCEWithLogitsLoss` и почему вторая безопаснее |  |
| Сформулировать разницу между loss и метрикой, привести пример недифференцируемости Accuracy |  |
| Реализовать вычисление Accuracy и F1 вручную на тензорах, без вызова sklearn внутри цикла |  |
| Собрать и обучить `FashionMLP` полным циклом: DataLoader -> Adam -> StepLR -> 10 эпох |  |
| Провести и объяснить эксперимент Adam vs SGD за первые эпохи |  |

## 11. Итоги Дня 4

**Что ты теперь знаешь:**

1. **`torch.optim`** заменяет ручное `p -= lr * p.grad` продвинутыми алгоритмами, которые решают проблемы единого learning rate и колебаний в оврагах функции потерь.
2. **SGD с momentum** накапливает экспоненциально взвешенную «инерцию» градиентов, сглаживая зигзаги и ускоряя движение в стабильном направлении. **Nesterov** делает это ещё точнее, «заглядывая вперёд».
3. **`weight_decay`** — это L2-регуляризация (Ridge), знакомая тебе по классическому ML, применённая к весам нейросети через сам оптимизатор, а не через явное изменение loss.
4. **Adam** поддерживает первый (`m`) и второй (`v`) момент градиента для **каждого параметра отдельно**, давая каждому параметру свой адаптивный эффективный `lr`. Bias correction компенсирует смещение оценок к нулю на первых шагах.
5. **Scheduler'ы** (`StepLR`, `ReduceLROnPlateau`) уменьшают `lr` по ходу обучения — большие шаги вначале, точная доводка ближе к минимуму.
6. **Gradient Clipping** (`clip_grad_norm_`) защищает от взрыва градиентов, масштабируя всю совокупность градиентов, сохраняя направление.
7. **`CrossEntropyLoss`** и **`BCEWithLogitsLoss`** реализуют численно стабильные версии формул, которые ты уже знал из классического ML (softmax + NLL, sigmoid + BCE), используя LogSumExp-трюк для защиты от переполнения — тот же принцип, что клиппинг вероятностей в твоей ручной логистической регрессии.
8. **Loss и метрика — разные сущности.** Loss — гладкий дифференцируемый суррогат для градиентного спуска; метрика — то, что реально важно, но часто недифференцируемо (Accuracy).

**Главный инсайт:** всё, что ты изучил сегодня, — это **инженерные решения** для проблем, с которыми ты уже сталкивался математически в предыдущих курсах: L2-регуляризация (Ridge), клиппинг вероятностей для численной стабильности логистической регрессии, `class_weight='balanced'` для дисбаланса классов. PyTorch не изобретает новую математику — он даёт готовые, оптимизированные и проверенные реализации того, что ты уже понимаешь на уровне формул.

**Связь с твоим roadmap:** в банковском ML (в том числе в FraudGuard) 90% моделей — градиентный бустинг, где своя система гиперпараметров (`learning_rate`, `num_leaves` и т.д. — уже знакомые тебе из курса по бустингу). Но сама концепция «loss как дифференцируемый суррогат бизнес-метрики» и «адаптивная скорость обучения» — это универсальные идеи ML-инженерии, которые пригодятся при обсуждении любого алгоритма на собеседовании, не только нейросетей.

**Переходи к Дню 5, когда:**
- Ты можешь объяснить разницу между `m_t` и `v_t` в Adam без подглядывания.
- Ты понимаешь, почему `lr=1e-3` — хороший дефолт для Adam, но не для SGD.
- Твой `FashionMLP` обучился за 10 эпох, показал ~87-89% test accuracy, и ты видишь ступенчатое падение `lr` на графике.
- Ты можешь объяснить, почему `CrossEntropyLoss` ожидает сырые логиты, а не вероятности после softmax.
- Ты провёл эксперимент Adam vs SGD и можешь объяснить результат, а не просто констатировать «Adam быстрее».

В Дне 5 ты откроешь «чёрный ящик» `DataLoader`, который использовал сегодня: напишешь собственный `Dataset` для табличных данных (кредитный скоринг), разберёшь `collate_fn`, `num_workers`, правильное разделение `train/val/test`, и — что особенно важно для банковских задач — `nn.Embedding` для категориальных признаков с высокой кардинальностью.